In [1]:
!pip install fredapi yfinance

In [5]:
import os
import re
import requests
import datetime
import numpy as np
import pandas as pd
import yfinance as yf
from fredapi import Fred
from scipy.optimize import minimize
import statsmodels.api as sm
from PIL import Image
import subprocess

# ==============================================================================
# 1. 自動從圖片精確提取 API 金鑰與設定
# ==============================================================================
try:
    # 嘗試安裝並使用 pytesseract 進行 OCR 辨識
    import pytesseract
except ImportError:
    subprocess.run(["pip", "install", "pytesseract"], capture_output=True)
    import pytesseract

try:
    # 讀取 /content/api_key.png 中的金鑰
    img = Image.open('/content/api_key.png')
    extracted_key = pytesseract.image_to_string(img).strip()

    # 使用正規表達式找出 32 碼的 16 進制金鑰 (FRED API 金鑰的標準格式)
    match = re.search(r'[a-fA-F0-9]{32}', extracted_key.replace(" ", ""))
    if match:
        FRED_API_KEY = match.group(0)
        print(f"[*] 已成功從 api_key.png 精確辨識並載入 FRED API 金鑰: {FRED_API_KEY[:4]}...{FRED_API_KEY[-4:]}")
    else:
        raise ValueError("在圖片中找不到 32 字元的 hex 格式 API Key")
except Exception as e:
    print(f"[!] 讀取圖片金鑰失敗: {e}，改嘗試從環境變數獲取")
    FRED_API_KEY = os.getenv("FRED_API_KEY", "828a95806ea5095791dfc681555143fb")

fred = Fred(api_key=FRED_API_KEY)

# 季度央行淨增持數據庫 (WGC Central Bank Net Purchases, tonnes)
# 每季由 WGC 報告公佈後更新一筆即可
CB_QUARTERLY_DB = {
    '2023Q2': 174.0, '2023Q3': 337.0, '2023Q4': 228.0,
    '2024Q1': 300.0, '2024Q2': 183.0, '2024Q3': 186.0, '2024Q4': 333.0,
    '2025Q1': 245.0, '2025Q2': 220.0, '2025Q3': 260.0, '2025Q4': 208.0,
    '2026Q1': 57.0, '2026Q2': 288.9, '2026Q3': 215.0
}


# ==============================================================================
# 2. 自動拉取月度高頻數據 (FRED + Yahoo Finance)
# ==============================================================================
def fetch_macro_data(start_date="2023-01-01"):
    print("[*] 正在向 Yahoo Finance 與 FRED 抓取最新宏觀數據...")

    # 1. 抓取 Yahoo Finance 資產 (黃金、美元指數、原油、SPDR GLD)
    tickers = {'Gold': 'GC=F', 'DXY': 'DX-Y.NYB', 'Brent': 'BZ=F', 'GLD': 'GLD'}
    raw_data = yf.download(list(tickers.values()), start=start_date, interval='1mo', progress=False)

    # 處理由於 yfinance 版本不同導致的欄位多重索引問題 (MultiIndex)
    if 'Adj Close' in raw_data.columns.levels[0] if isinstance(raw_data.columns, pd.MultiIndex) else 'Adj Close' in raw_data.columns:
        raw_yf = raw_data['Adj Close']
    else:
        raw_yf = raw_data['Close']

    raw_yf = raw_yf.copy()
    raw_yf.rename(columns={v: k for k, v in tickers.items()}, inplace=True)
    raw_yf.index = pd.to_datetime(raw_yf.index).to_period('M').to_timestamp()

    # 2. 抓取 FRED 10 年期實質殖利率 (DFII10: 10-Year TIPS)
    real_yield = fred.get_series('DFII10', observation_start=start_date)
    real_yield = real_yield.resample('MS').last()  # 取每月末數值
    real_yield.name = 'RealYield'

    # 3. 欄位合併與特徵轉換
    df = raw_yf.merge(real_yield, left_index=True, right_index=True, how='inner')

    df['Gold_pct'] = np.log(df['Gold'] / df['Gold'].shift(1)) * 100
    df['DXY_pct'] = np.log(df['DXY'] / df['DXY'].shift(1)) * 100
    df['Brent_pct'] = np.log(df['Brent'] / df['Brent'].shift(1)) * 100
    df['RealYield_bps'] = (df['RealYield'] - df['RealYield'].shift(1)) * 100
    # GLD 持倉成交量估算流量
    df['ETF_flow'] = (df['GLD'] - df['GLD'].shift(1)) / df['GLD'].shift(1) * 100

    return df


# ==============================================================================
# 3. 混頻特徵工程與 t-2 季央行資料對齊
# ==============================================================================
def prepare_midas_dataset(df_monthly):
    # 構建季度 DataFrame 並對齊
    q_df = pd.DataFrame(list(CB_QUARTERLY_DB.items()), columns=['Quarter', 'CB_purchases'])
    q_df['Timestamp'] = pd.PeriodIndex(q_df['Quarter'], freq='Q').to_timestamp()
    q_df.set_index('Timestamp', inplace=True)

    # 嚴格施加 t-2 季與 t-3 季滯後
    q_df['CB_lag2'] = q_df['CB_purchases'].shift(2)
    q_df['CB_lag3'] = q_df['CB_purchases'].shift(3)

    # 複製 DataFrame 並保留原本的 Index 名稱
    df_monthly = df_monthly.copy()
    original_index_name = df_monthly.index.name if df_monthly.index.name is not None else 'Date'
    df_monthly.index.name = 'original_date_index'

    df_monthly['Quarter'] = df_monthly.index.to_period('Q').to_timestamp()
    merged = df_monthly.reset_index().merge(
        q_df[['CB_lag2', 'CB_lag3']], left_on='Quarter', right_index=True, how='left'
    ).set_index('original_date_index')

    merged.index.name = original_index_name
    merged.drop(columns=['Quarter'], inplace=True)

    # 高頻特徵滯後 1 期 (預測 t 月需使用 t-1 月數據)
    for col in ['DXY_pct', 'RealYield_bps', 'Brent_pct', 'ETF_flow']:
        merged[f'{col}_lag1'] = merged[col].shift(1)

    return merged.dropna(subset=['DXY_pct_lag1', 'CB_lag2', 'CB_lag3'])


# ==============================================================================
# 4. 指數 Almon MIDAS 模型與預測引擎
# ==============================================================================
def almon_weights(theta1, theta2, lags=2):
    j = np.arange(1, lags + 1)
    power = theta1 * j + theta2 * (j ** 2)
    power = power - np.max(power)
    w = np.exp(power)
    return w / np.sum(w)


def midas_loss(params, X_m, Z_q, y):
    alpha = params[0]
    beta_m = params[1:5]
    gamma_cb = params[5]
    theta1, theta2 = params[6], params[7]
    w = almon_weights(theta1, theta2, lags=Z_q.shape[1])
    y_hat = alpha + np.dot(X_m, beta_m) + gamma_cb * np.dot(Z_q, w)
    return np.sum((y - y_hat) ** 2)


def run_forward_inference(df):
    feature_m = ['DXY_pct_lag1', 'RealYield_bps_lag1', 'Brent_pct_lag1', 'ETF_flow_lag1']
    feature_q = ['CB_lag2', 'CB_lag3']

    # 訓練集：歷史已實現的月度數據 (Gold_pct 非空值)
    train_df = df[df['Gold_pct'].notnull()].copy()

    y_train = train_df['Gold_pct'].values
    X_m_train = train_df[feature_m].values
    Z_q_train = train_df[feature_q].values

    # NLS 估計最佳參數
    init_params = np.array([0.5, -0.8, -0.05, 0.1, 0.3, 0.01, 0.0, -0.01])
    res = minimize(midas_loss, init_params, args=(X_m_train, Z_q_train, y_train), method='L-BFGS-B')
    opt = res.x

    # 最新當月預測特徵向量 (位於最後一列)
    latest_row = df.iloc[-1]
    X_m_latest = latest_row[feature_m].values.astype(float)
    Z_q_latest = latest_row[feature_q].values.astype(float)

    w_opt = almon_weights(opt[6], opt[7], lags=2)
    pred_return = opt[0] + np.dot(X_m_latest, opt[1:5]) + opt[5] * np.dot(Z_q_latest, w_opt)

    return pred_return, latest_row.name, opt


# ==============================================================================
# 5. 主流程與報表產出
# ==============================================================================
if __name__ == "__main__":
    df_raw = fetch_macro_data()
    dataset = prepare_midas_dataset(df_raw)

    pred_pct, target_date, opt_coeffs = run_forward_inference(dataset)

    direction = "看漲 (Bullish 🟢)" if pred_pct > 0 else "看跌 (Bearish 🔴)"
    report = f"""
==================================================
  MIDAS 黃金月度定量預測報告 ({target_date.strftime('%Y年%m月')})
==================================================
• 預估月度漲跌幅: {pred_pct:+.2f}%
• 模型信號定性: {direction}
--------------------------------------------------
[主要因子最新驅動狀態 (Lagged)]
• 美元指數變動 (DXY): {dataset['DXY_pct_lag1'].iloc[-1]:+.2f}%
• 10Y TIPS 實質利率變動: {dataset['RealYield_bps_lag1'].iloc[-1]:+.1f} bps
• 布蘭特原油變動: {dataset['Brent_pct_lag1'].iloc[-1]:+.2f}%
• 央行購金基準 (t-2季): {dataset['CB_lag2'].iloc[-1]:.1f} 公噸
==================================================
"""
    print(report)

[*] 已成功從 api_key.png 精確辨識並載入 FRED API 金鑰: 828a...43fb
[*] 正在向 Yahoo Finance 與 FRED 抓取最新宏觀數據...


/tmp/ipykernel_1112/870733071.py:60: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(list(tickers.values()), start=start_date, interval='1mo', progress=False)



  MIDAS 黃金月度定量預測報告 (2026年09月)
• 預估月度漲跌幅: +5.28%
• 模型信號定性: 看漲 (Bullish 🟢)
--------------------------------------------------
[主要因子最新驅動狀態 (Lagged)]
• 美元指數變動 (DXY): -0.37%
• 10Y TIPS 實質利率變動: -3.0 bps
• 布蘭特原油變動: +0.41%
• 央行購金基準 (t-2季): 57.0 公噸

